# Librispeech Yield Investigation
Use the slider to see how many usable virtual clips (and speakers) are generated based on your requested `signal_lengths`.

In [ ]:
import os
import sys
from pathlib import Path
import torch
import torchaudio
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import ipywidgets as widgets
from IPython.display import display, HTML

plt.style.use('seaborn-v0_8-darkgrid')

# Setup project root
PROJECT_ROOT = Path(os.getcwd()).parent
os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))


In [ ]:
LIBRISPEECH_DIR = Path("/data4/Henri/MuSE-Toolbox/data/databases/librispeech")
CACHE_FILE = LIBRISPEECH_DIR / "lengths_cache.pt"

splits = ["train-clean-360", "dev-clean", "test-clean"]

# A dictionary: dict[split][speaker][chapter] = [list of utterance lengths in seconds]
if CACHE_FILE.exists():
    print(f"Loading pre-computed file lengths from {CACHE_FILE}")
    db_lengths = torch.load(CACHE_FILE)
else:
    print("Extracting file lengths from all Librispeech files... (this will take a few minutes the first time)")
    db_lengths = {s: {} for s in splits}
    
    for split in splits:
        split_path = LIBRISPEECH_DIR / split
        if not split_path.exists(): continue
            
        speakers = [d for d in split_path.iterdir() if d.is_dir()]
        for spk_path in tqdm(speakers, desc=f"Scanning {split}"):
            db_lengths[split][spk_path.name] = {}
            for chap_path in spk_path.iterdir():
                if not chap_path.is_dir(): continue
                    
                files = sorted(chap_path.glob("*.flac"))
                lengths = []
                last_id = -1
                
                for f in files:
                    utt_id = int(f.stem.split("-")[-1])
                    info = torchaudio.info(f)
                    dur = info.num_frames / info.sample_rate
                    # Store (utterance_id, duration) to check for consecutive later
                    lengths.append((utt_id, dur))
                    
                db_lengths[split][spk_path.name][chap_path.name] = lengths
                
    torch.save(db_lengths, CACHE_FILE)
    print("Done scanning and caching!")


In [ ]:
def analyze_yield(signal_length):
    # The database target duration logic is now 1.5 * signal_length capped at 170s
    target_duration = min(1.5 * signal_length, 170.0)
    
    results = {}
    total_clips_all_splits = 0
    
    for split, speakers in db_lengths.items():
        total_speakers = len(speakers)
        valid_speakers = 0
        total_clips = 0
        
        for spk, chapters in speakers.items():
            spk_has_clip = False
            for chap, utterances in chapters.items():
                current_duration = 0.0
                last_id = -1
                
                for i, (utt_id, dur) in enumerate(utterances):
                    is_consecutive = (last_id != -1) and (utt_id == last_id + 1)
                    if not is_consecutive and i > 0:
                        current_duration = 0.0  # continuity broken
                        
                    current_duration += dur
                    last_id = utt_id
                    
                    if current_duration >= target_duration:
                        total_clips += 1
                        spk_has_clip = True
                        current_duration = 0.0
                        last_id = -1
                        
            if spk_has_clip:
                valid_speakers += 1
                
        results[split] = {
            "Total Speakers": total_speakers,
            "Valid Speakers": valid_speakers,
            "Total Virtual Clips generated": total_clips
        }
        total_clips_all_splits += total_clips
        
    return results, total_clips_all_splits

# --- Interactive Widget ---
w_signal_length = widgets.FloatSlider(value=60.0, min=10.0, max=1200.0, step=10.0, description="Signal Len (s):", layout={'width': '500px'})
out = widgets.Output()

def update_ui(change):
    with out:
        out.clear_output()
        length = w_signal_length.value
        target = min(length * 1.5, 170.0)
        
        display(HTML(f"<h3>Analysis for signal_lengths = {length}s</h3>"))
        display(HTML(f"<i>(Note: This requires consecutive utterances summing to a target_duration of min(1.5 * length, 170) = {target}s)</i><br><br>"))
        
        res, total = analyze_yield(length)
        
        html = "<table border='1' style='border-collapse: collapse; text-align: left;'>"
        html += "<tr><th style='padding: 8px;'>Split</th><th style='padding: 8px;'>Original Speakers</th><th style='padding: 8px;'>Usable Speakers</th><th style='padding: 8px;'>Virtual Clips Generated</th></tr>"
        
        for split, data in res.items():
            if data['Total Speakers'] == 0: continue
            pct_spk = (data['Valid Speakers'] / data['Total Speakers']) * 100
            html += f"<tr><td style='padding: 8px;'><b>{split}</b></td>"
            html += f"<td style='padding: 8px;'>{data['Total Speakers']}</td>"
            html += f"<td style='padding: 8px;'>{data['Valid Speakers']} <b>({pct_spk:.1f}%)</b></td>"
            html += f"<td style='padding: 8px;'>{data['Total Virtual Clips generated']}</td></tr>"
            
        html += "</table>"
        display(HTML(html))
        
        if total == 0:
            display(HTML(f"<br><b style='color:red;'>WARNING: This signal length is too long! Your precomputation step will yield ZERO data.</b>"))
        else:
            display(HTML(f"<br><b style='color:green;'>SUCCESS: You have {total} total virtual clips to draw from!</b>"))

w_signal_length.observe(update_ui, names='value')
display(widgets.VBox([w_signal_length, out]))
update_ui(None)
